# conv-output-shape composite — cx10: predict + verify conv2d output shape when padding is non-zero

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-output-shape`, `conv-padding-zero`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-output-shape"
DD_ATOM_IDS = ["conv-output-shape", "conv-padding-zero"]
DD_SUBTOPICS = ["CNN: Conv output shape", "CNN: Conv zero padding"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The 2-D conv output-shape formula, with stride and padding, is:
```
H_out = (H + 2*PH - KH) // SH + 1
W_out = (W + 2*PW - KW) // SW + 1
```

The `2*P` term comes directly from the `conv-padding-zero` atom: padding by `P` on each side adds `2*P` to the effective input length BEFORE the kernel walks over it. The `// S + 1` comes from `conv-stride-downsample` arithmetic.

**Same-padding inversion.** For stride 1 and odd kernel `K`, the choice `P = (K - 1) // 2` gives `H_out == H` (the input shape is preserved). This drill exercises BOTH directions:
- Forward: given hyperparams, predict `(B, OC, H_out, W_out)`.
- Inverse: given `K` (odd, stride-1), compute the `P` that makes `H_out == H` (the 'same' padding).

The forward direction tests the `conv-output-shape` atom with non-zero padding; the inverse tests understanding that padding cancels kernel-shrink for the same-padding regime.

### Composite Exercise — predict + verify conv2d output shape when padding is non-zero

**Atoms exercised together**: `conv-output-shape`, `conv-padding-zero`

Implement two functions.

1. `cx10_outshape_with_pad(input_shape, out_channels, kernel_size, stride, padding)`:
   - `input_shape`: `(B, IC, H, W)`.
   - `kernel_size`, `stride`, `padding`: each a 2-tuple `(h_val, w_val)`.
   - Return `(B, OC, H_out, W_out)` per the formula above.

2. `cx10_same_padding(kernel_size)`:
   - `kernel_size`: 2-tuple `(KH, KW)`. Each must be odd.
   - Return `(PH, PW)` = `((KH - 1) // 2, (KW - 1) // 2)`.
   - For stride 1, applying this padding makes `H_out == H` and `W_out == W` (verified by the test).

The test:
- Forward-checks `cx10_outshape_with_pad` against a real `nn.Conv2d`.
- Inversely checks that `cx10_same_padding` paired with stride-1 yields an identity-shape conv.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx10_outshape_with_pad(input_shape, out_channels, kernel_size, stride, padding):
    raise NotImplementedError

def cx10_same_padding(kernel_size):
    raise NotImplementedError

def _test_cx10():
    from torch import nn

    def _check_forward(input_shape, oc, k, s, p):
        predicted = cx10_outshape_with_pad(input_shape, oc, k, s, p)
        conv = nn.Conv2d(
            in_channels=input_shape[1], out_channels=oc,
            kernel_size=k, stride=s, padding=p,
        )
        x = t.zeros(*input_shape)
        actual = tuple(conv(x).shape)
        assert tuple(predicted) == actual, (
            f'shape mismatch for {input_shape} oc={oc} k={k} s={s} p={p}:\n'
            f'  predicted {tuple(predicted)}\n  actual    {actual}'
        )

    # Case A: stride-1 same-padding (odd kernel) — output equals input on H, W.
    _check_forward((1, 3, 32, 32), 16, (3, 3), (1, 1), (1, 1))   # 3,3 same-pad
    _check_forward((1, 3, 32, 32), 16, (5, 5), (1, 1), (2, 2))   # 5,5 same-pad
    _check_forward((1, 3, 32, 32), 16, (7, 7), (1, 1), (3, 3))   # 7,7 same-pad

    # Case B: padding bigger than the kernel — output strictly larger than input.
    _check_forward((1, 3, 8, 8), 1, (3, 3), (1, 1), (2, 2))

    # Case C: non-square padding and kernel.
    _check_forward((2, 1, 28, 40), 4, (5, 3), (2, 1), (2, 1))

    # Case D: zero padding (no-op) — degenerates to (H-K)//S + 1.
    _check_forward((1, 3, 32, 32), 16, (3, 3), (1, 1), (0, 0))
    _check_forward((1, 3, 32, 32), 16, (3, 3), (2, 2), (0, 0))

    # Case E: stride > 1 with padding — the stride-2 + same-padding ResNet pattern.
    _check_forward((4, 8, 64, 64), 32, (3, 3), (2, 2), (1, 1))   # halves spatial axes cleanly
    _check_forward((1, 1, 16, 16), 1, (5, 5), (2, 2), (2, 2))

    # Case F: same-padding inversion — for each odd kernel, the helper produces P that makes
    # H_out == H (stride 1).
    for K in [1, 3, 5, 7, 9]:
        P = cx10_same_padding((K, K))
        assert P == ((K - 1) // 2, (K - 1) // 2), f'K={K}: same_padding wrong, got {P}'
        out_shape = cx10_outshape_with_pad((1, 1, 24, 24), 1, (K, K), (1, 1), P)
        assert tuple(out_shape) == (1, 1, 24, 24), (
            f'same-pad K={K} P={P} should give (1,1,24,24), got {tuple(out_shape)}'
        )

    # Case G: cross-check same-padding helper against nn.Conv2d for non-square odd kernels.
    for KH, KW in [(3, 5), (5, 1), (1, 7), (7, 7)]:
        P = cx10_same_padding((KH, KW))
        conv = nn.Conv2d(3, 8, kernel_size=(KH, KW), stride=1, padding=P)
        x = t.zeros(1, 3, 20, 20)
        y = conv(x)
        assert tuple(y.shape) == (1, 8, 20, 20), (
            f'same-pad failed: KH={KH} KW={KW} P={P} produced {tuple(y.shape)}, expected (1,8,20,20)'
        )
    _dd_passed.add('cx10')

_test_cx10()

<details><summary>Show solution — cx10</summary>

```python
def cx10_outshape_with_pad(input_shape, out_channels, kernel_size, stride, padding):
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    # Atom A (conv-output-shape) with non-zero padding term:
    H_out = (H + 2 * PH - KH) // SH + 1
    W_out = (W + 2 * PW - KW) // SW + 1
    return (B, out_channels, H_out, W_out)

def cx10_same_padding(kernel_size):
    KH, KW = kernel_size
    # Atom B (conv-padding-zero): the amount of zero-pad needed so the kernel center can
    # reach every input cell — (K - 1) // 2 per side for stride 1, odd K.
    return ((KH - 1) // 2, (KW - 1) // 2)
```

The `2*P` term in the output-shape formula directly reflects the `conv-padding-zero` atom: each side contributes `P` extra cells the kernel can land on. Same-padding (`P = (K-1)//2`) exactly cancels the `K - 1` shrink for stride 1 — that's why ResNet's 3x3 + padding=1 keeps spatial dims constant. For stride > 1, same-padding ALONE doesn't preserve shape; it only ensures the formula divides cleanly. Forgetting the `2*` (writing `P` instead of `2*P`) is the canonical off-by-one bug here.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["CNN: Conv output shape", "CNN: Conv zero padding"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()